Используем selenium для парсинга сайта hirify

In [ ]:
!pip install selenium

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import logging
import sys
import pandas as pd

https://discourse.jupyter.org/t/jupyter-does-not-print-logging-information/11790

In [104]:
BASE_URL = "https://hirify.me"
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger('selenium_scrapper')
logger.setLevel(logging.DEBUG)
def prepare_driver_page(period: str = "?period=week") -> webdriver.Chrome:
    driver = webdriver.Chrome()
    logger.info("Driver created")
    driver.get(BASE_URL + period)
    time.sleep(2)
    logger.info("Page loaded")
    settings_button = driver.find_element(By.CLASS_NAME, "settings-button")
    settings_button.click()
    filter_up = driver.find_element(By.CLASS_NAME, "filters-row")
    filters = filter_up.find_elements(By.TAG_NAME, "input")
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
        if not f.is_selected():
            f.click()
    for f in filters:
        logger.debug(f"Filter {f.get_attribute('id')} is selected: {f.is_selected()}")
    logger.info("Filters applied")
    return driver


driver = prepare_driver_page()

2026-04-07 02:20:04,516 Driver created
2026-04-07 02:20:11,682 Page loaded
2026-04-07 02:20:11,785 Filter  is selected: True
2026-04-07 02:20:11,795 Filter  is selected: False
2026-04-07 02:20:11,844 Filter  is selected: True
2026-04-07 02:20:11,853 Filter  is selected: False
2026-04-07 02:20:11,898 Filter  is selected: True
2026-04-07 02:20:11,904 Filter  is selected: True
2026-04-07 02:20:11,911 Filter  is selected: True
2026-04-07 02:20:11,918 Filter  is selected: True
2026-04-07 02:20:11,918 Filters applied


In [105]:
def parse_cards(driver: webdriver.Chrome):
    cards = driver.find_elements(By.CSS_SELECTOR, "div.vacancy-card[data-vacancy-id]")
    logger.info(f"Found {len(cards)} cards")
    return cards

cards = parse_cards(driver)

2026-04-07 02:20:14,473 Found 15 cards


In [108]:
def parse_vacancys(cards: list) -> list[dict]:
    vacancys = []
    for card in cards:
        logger.info(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
        vacancy_link = card.find_element(By.CLASS_NAME, "vacancy-card-link")
        vacancy_title = card.find_element(By.CLASS_NAME, "title")
        vacancy_tags_div = card.find_element(By.CLASS_NAME, "common-tags")
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
        try:
            vacancy_company = card.find_element(By.CLASS_NAME, "company")
        except Exception as e:
            vacancy_company = None
        vacancy_time = card.find_element(By.CLASS_NAME, "date")
        vacancy_skills_div = card.find_element(By.CLASS_NAME, "vacancy-tags")
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
        try:
            vacansy_salary = card.find_element(By.CLASS_NAME, "salary")
        except Exception as e:
            vacansy_salary = None
        vacancys.append(
            {
                "title": vacancy_title.text,
                "link": vacancy_link.get_attribute("href"),
                "tags": vacancy_tags,
                "company": vacancy_company.text if vacancy_company else "No company",
                "time": vacancy_time.text,
                "skills": vacancy_skills,
                "salary": vacansy_salary.text if vacansy_salary else "No salary",
            }
        )
    return vacancys
vacancys = parse_vacancys(cards)
print(vacancys[0])

2026-04-07 02:20:25,007 Parsing card with id 440455
2026-04-07 02:20:25,133 Parsing card with id 440454
2026-04-07 02:20:25,218 Parsing card with id 440453
2026-04-07 02:20:25,311 Parsing card with id 440452
2026-04-07 02:20:25,423 Parsing card with id 440451
2026-04-07 02:20:25,539 Parsing card with id 440450
2026-04-07 02:20:25,644 Parsing card with id 440449
2026-04-07 02:20:25,738 Parsing card with id 440448
2026-04-07 02:20:25,854 Parsing card with id 440447
2026-04-07 02:20:25,967 Parsing card with id 440446
2026-04-07 02:20:26,062 Parsing card with id 440445
2026-04-07 02:20:26,159 Parsing card with id 440444
2026-04-07 02:20:26,276 Parsing card with id 440443
2026-04-07 02:20:26,385 Parsing card with id 440442
2026-04-07 02:20:26,538 Parsing card with id 440441
{'title': 'Head Of Experience Design (AI)', 'link': 'https://hirify.me/jobs/440455-head-experience-design-ai', 'tags': ['remote (Europe)', 'Spain', 'fulltime', 'head'], 'company': 'Company hidden', 'time': '29 seconds ag

Переписываем все это на 3 функции: 1 итератор по стринице 2 парсер 1 карточки 3 безопасные getter 

In [117]:
def safe_find_element(parent, by, value):
    try:
        return parent.find_element(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements(parent, by, value):
    try:
        return parent.find_elements(by, value)
    except Exception as e:
        logger.debug(f"Element not found: {e}")
        return None

def safe_find_elements_text(parent, by, value):
    r = safe_find_elements(parent, by, value)
    if r:
        if isinstance(r, list):
            return [elem.text for elem in r]
        else:
            return r.text
    return None

def parse_card(card):
    logger.debug(f"Parsing card with id {card.get_attribute('data-vacancy-id')}") # Используем get_attribute на случай если у элемента нет детей
    vacancy_link = safe_find_element(card, By.CLASS_NAME, "vacancy-card-link")
    vacancy_title = safe_find_element(card, By.CLASS_NAME, "title")
    vacancy_tags_div = safe_find_element(card, By.CLASS_NAME, "common-tags")
    vacancy_tags = []
    if vacancy_tags_div:
        vacancy_tags = vacancy_tags_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_tags = [t.text for t in vacancy_tags]
    vacancy_company = safe_find_element(card, By.CLASS_NAME, "company").text
    vacancy_time = safe_find_element(card, By.CLASS_NAME, "date")
    vacancy_skills_div = safe_find_element(card, By.CLASS_NAME, "vacancy-tags")
    vacancy_skills = []
    if vacancy_skills_div:
        vacancy_skills = vacancy_skills_div.find_elements(By.CLASS_NAME, "tag")
        vacancy_skills = [s.text for s in vacancy_skills]
    vacansy_salary = safe_find_elements_text(card, By.CLASS_NAME, "salary")
    return {
        "title": vacancy_title.text,
        "link": vacancy_link.get_attribute("href"),
        "tags": vacancy_tags,
        "company": vacancy_company,
        "time": vacancy_time.text,
        "skills": vacancy_skills,
        "salary": vacansy_salary
    }

def parse_vacancys(cards: list) -> list[dict]:
    vacancies = []
    for card in cards:
        parse_card(card)
        vacancies.append(parse_card(card))
    return vacancies

vacancies = parse_vacancys(cards)
df = pd.DataFrame(vacancies)
df.head(20)

2026-04-07 02:29:58,068 Parsing card with id 440455
2026-04-07 02:29:58,162 Parsing card with id 440455
2026-04-07 02:29:58,256 Parsing card with id 440454
2026-04-07 02:29:58,351 Parsing card with id 440454
2026-04-07 02:29:58,449 Parsing card with id 440453
2026-04-07 02:29:58,529 Parsing card with id 440453
2026-04-07 02:29:58,618 Parsing card with id 440452
2026-04-07 02:29:58,705 Parsing card with id 440452
2026-04-07 02:29:58,796 Parsing card with id 440451
2026-04-07 02:29:58,896 Parsing card with id 440451
2026-04-07 02:29:59,005 Parsing card with id 440450
2026-04-07 02:29:59,110 Parsing card with id 440450
2026-04-07 02:29:59,211 Parsing card with id 440449
2026-04-07 02:29:59,298 Parsing card with id 440449
2026-04-07 02:29:59,401 Parsing card with id 440448
2026-04-07 02:29:59,533 Parsing card with id 440448
2026-04-07 02:29:59,656 Parsing card with id 440447
2026-04-07 02:29:59,750 Parsing card with id 440447
2026-04-07 02:29:59,841 Parsing card with id 440446
2026-04-07 0

,title,link,tags,company,time,skills,salary
0,Head Of Experience Design (AI),https://hirify.me/jobs/440455-head-experience-...,"[remote (Europe), Spain, fulltime, head]",Company hidden,29 seconds ago,"[ai, ux, ui, product design, design system, +1...",None
1,Global Hr Operations Lead,https://hirify.me/jobs/440454-global-hr-operat...,"[remote (Europe)/hybrid, Armenia, fulltime, lead]",Company hidden,37 seconds ago,"[excel, automation, labor law, international h...",None
2,Senior Product Marketing Manager (SaaS),https://hirify.me/jobs/440453-senior-product-m...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,37 seconds ago,"[saas, product marketing, stakeholder manageme...",None
3,Senior Backend Engineer (Platform),https://hirify.me/jobs/440452-senior-backend-e...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,45 seconds ago,"[ai, java, kotlin, javascript, typescript, +6 ...",None
4,Talent Acquisition (Gamedev),https://hirify.me/jobs/440451-talent-acquisiti...,"[remote (USA), US, fulltime, middle]",Company hidden,51 seconds ago,"[recruiting, jira, confluence, talent acquisit...",None
5,Vp Marketing (SaaS),https://hirify.me/jobs/440450-vp-marketing-saas,"[remote (Global), Germany, fulltime, director]",Company hidden,1 minute ago,"[ai, seo, saas, product marketing, brand posit...",None
6,Senior Backend Engineer (Fintech),https://hirify.me/jobs/440449-senior-backend-e...,"[remote (Europe), Spain, fulltime, senior]",Company hidden,1 minute ago,"[ai, java, kotlin, kafka, linux, +4 skills]",None
7,Paid Acquisition Manager,https://hirify.me/jobs/440448-paid-acquisition...,"[remote (Europe), Spain, fulltime, middle]",Company hidden,1 minute ago,"[ai, excel, english, performance marketing, go...",None
8,Senior Machine Learning Engineer (AI),https://hirify.me/jobs/440447-senior-machine-l...,"[Poland, fulltime, senior]",Company hidden,2 minutes ago,"[ai, python, kubernetes, redis, kafka, +5 skills]",None
9,People Operations AI & Analytics Partner,https://hirify.me/jobs/440446-people-operation...,"[remote (Global), Germany, fulltime]",Company hidden,2 minutes ago,"[ai, data analysis, automation, people operati...",None


Итак погинация